In [1]:
import sklearn
import squarify
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from datetime import datetime, timedelta

In [2]:
%reload_ext watermark
%watermark -a "Matheus dos Anjos" --iversions

Author: Matheus dos Anjos

seaborn   : 0.13.2
sklearn   : 1.5.1
matplotlib: 3.9.2
squarify  : 0.4.4
pandas    : 2.2.2
plotly    : 5.24.1



In [3]:
df = pd.read_csv("C:/Users/conta/OneDrive/Desktop/Projetos-/Projetos Python/Projetos de Machine Learning/dataset.csv")

In [4]:
df.shape 

(116581, 53)

In [5]:
df.head()

,order_id,order_id3,customer_id3,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,...,seller_city,seller_state,product_category_name_english,review_response_time,order_purchase_year,order_purchase_month,order_purchase_dayofweek,order_purchase_hour,order_purchase_day,order_purchase_mon
0,ON34305,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,...,Maua,SP,housewares,1.0,2017,10,0,10,Mon,Oct
1,ON34305,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,...,Maua,SP,housewares,1.0,2017,10,0,10,Mon,Oct
2,ON34305,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,...,Maua,SP,housewares,1.0,2017,10,0,10,Mon,Oct
3,ON40291,128e10d95713541c87cd1a2e48201934,a20e8105f23924cd00833fd87daa0831,delivered,2017-08-15 18:29:31,2017-08-15 20:05:16,2017-08-17 15:28:33,2017-08-18 14:44:43,2017-08-28 00:00:00,1.0,...,Maua,SP,housewares,1.0,2017,8,1,18,Tue,Aug
4,ON74313,0e7e841ddf8f8f2de2bad69267ecfbcf,26c7ac168e1433912a51b924fbd34d34,delivered,2017-08-02 18:24:47,2017-08-02 18:43:15,2017-08-04 17:35:43,2017-08-07 18:30:01,2017-08-15 00:00:00,1.0,...,Maua,SP,housewares,0.0,2017,8,2,18,Wed,Aug


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116581 entries, 0 to 116580
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       116581 non-null  object 
 1   order_id3                      116581 non-null  object 
 2   customer_id3                   116581 non-null  object 
 3   order_status                   116581 non-null  object 
 4   order_purchase_timestamp       116581 non-null  object 
 5   order_approved_at              116581 non-null  object 
 6   order_delivered_carrier_date   116581 non-null  object 
 7   order_delivered_customer_date  116581 non-null  object 
 8   order_estimated_delivery_date  116581 non-null  object 
 9   order_item_id                  116581 non-null  float64
 10  product_id3                    116581 non-null  object 
 11  seller_id3                     116581 non-null  object 
 12  shipping_limit_date           

In [7]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

In [8]:
df['order_purchase_date'] = df['order_purchase_timestamp'].dt.date

In [9]:
df['InvoiceDate'] = df.order_purchase_date.apply(lambda x: datetime.strftime(x, '%Y-%m-%d'))
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [10]:
df['customer_id'].isnull().sum()

0

In [11]:
df['InvoiceDate'].min()

Timestamp('2016-09-04 00:00:00')

In [12]:
df['InvoiceDate'].max()

Timestamp('2018-09-03 00:00:00')

In [13]:
data_mais_recente = df['InvoiceDate'].max() + timedelta(days=1)
data_mais_recente

Timestamp('2018-09-04 00:00:00')

In [14]:
agregacao = df.groupby(['customer_unique_id']).agg({'InvoiceDate': lambda x: (data_mais_recente - x.max()).days,
                                                    'order_id': 'count',
                                                    'payment_value': 'sum'})

In [15]:
agregacao.rename(columns= {'InvoiceDate': 'Recenty',
                           'order_id': 'Frequency',
                           'payment_value': 'Monetary'},
                           inplace=True)

In [16]:
agregacao.head()

,Recenty,Frequency,Monetary
customer_unique_id,,,
C00001,73,1,121.82
C00002,66,1,155.76
C00003,337,1,181.55
C00004,193,1,90.78
C00005,283,1,266.89


In [17]:
agregacao.isnull().sum()

Recenty      0
Frequency    0
Monetary     0
dtype: int64

In [18]:
agregacao.describe()

,Recenty,Frequency,Monetary
count,94087.000000,94087.000000,94087.000000
mean,243.803575,1.239077,214.249054
std,153.156983,0.850594,647.368039
min,1.000000,1.000000,9.590000
25%,120.000000,1.000000,64.000000
50%,224.000000,1.000000,113.150000
75%,353.000000,1.000000,203.770000
max,730.000000,75.000000,109312.640000


In [19]:
r_labels = range(4, 0, -1)

In [20]:
r_groups = pd.qcut(agregacao['Recenty'], q = 4, labels = r_labels)

In [21]:
r_groups.value_counts()

Recenty
4    23766
2    23563
1    23440
3    23318
Name: count, dtype: int64

In [22]:
agregacao = agregacao.assign(R = r_groups.values)

In [23]:
agregacao.sample(5)

,Recenty,Frequency,Monetary,R
customer_unique_id,,,,
C33763,273,1,189.72,2
C70607,275,1,63.73,2
C81743,264,1,116.23,2
C65705,329,1,120.09,2
C35676,183,1,101.34,3


In [24]:
agregacao['Frequency'].value_counts()

Frequency
1     79915
2     10323
3      2003
4       956
5       337
6       300
7        73
8        45
10       25
9        25
12       25
11       18
14        8
15        7
24        7
20        4
13        4
21        3
22        1
38        1
26        1
75        1
18        1
19        1
29        1
16        1
35        1
Name: count, dtype: int64

In [25]:
f_labels = range(1, 3)

In [26]:
def pct_rank_qcut(series, n):

    edges = pd.Series([float(i) / n for i in range(n + 1)])

    f = lambda x: (edges >= x).values.argmax()

    return series.rank(pct = 1).apply(f)

In [27]:
f_groups = pct_rank_qcut(agregacao['Frequency'], 2)

In [28]:
f_groups.value_counts()

Frequency
1    79915
2    14172
Name: count, dtype: int64

In [29]:
agregacao = agregacao.assign(F= f_groups.values)

In [30]:
agregacao.sample(10)

,Recenty,Frequency,Monetary,R,F
customer_unique_id,,,,,
C74141,299,1,224.48,2,1
C31134,296,1,68.02,2,1
C02976,433,1,33.09,1,1
C81287,312,1,111.90,2,1
C13941,300,1,444.92,2,1
C72852,408,3,450.89,1,2
C65516,190,1,64.17,3,1
C75812,400,1,70.72,1,1
C07625,190,2,331.92,3,2


In [31]:
m_labels = range(1, 5)

In [32]:
m_groups = pd.qcut(agregacao['Monetary'], q = 4, labels = m_labels)

In [33]:
m_groups.value_counts()

Monetary
1    23569
4    23521
3    23503
2    23494
Name: count, dtype: int64

In [34]:
agregacao = agregacao.assign(M = m_groups.values)

In [35]:
agregacao.head()

,Recenty,Frequency,Monetary,R,F,M
customer_unique_id,,,,,,
C00001,73,1,121.82,4,1,3
C00002,66,1,155.76,4,1,3
C00003,337,1,181.55,2,1,3
C00004,193,1,90.78,3,1,2
C00005,283,1,266.89,2,1,4


In [36]:
agregacao.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94087 entries, C00001 to C96999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Recenty    94087 non-null  int64   
 1   Frequency  94087 non-null  int64   
 2   Monetary   94087 non-null  float64 
 3   R          94087 non-null  category
 4   F          94087 non-null  int64   
 5   M          94087 non-null  category
dtypes: category(2), float64(1), int64(3)
memory usage: 3.8+ MB


In [37]:
le = LabelEncoder()

In [38]:
agregacao['R'] = le.fit_transform(agregacao['R']) + 1
agregacao['M'] = le.fit_transform(agregacao['M']) + 1

In [39]:
agregacao.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94087 entries, C00001 to C96999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Recenty    94087 non-null  int64  
 1   Frequency  94087 non-null  int64  
 2   Monetary   94087 non-null  float64
 3   R          94087 non-null  int64  
 4   F          94087 non-null  int64  
 5   M          94087 non-null  int64  
dtypes: float64(1), int64(5)
memory usage: 7.0+ MB


In [40]:
agregacao['Score_RFM'] = agregacao[['R', 'F', 'M']].sum(axis = 1)

In [41]:
agregacao.head()

,Recenty,Frequency,Monetary,R,F,M,Score_RFM
customer_unique_id,,,,,,,
C00001,73,1,121.82,4,1,3,8
C00002,66,1,155.76,4,1,3,8
C00003,337,1,181.55,2,1,3,6
C00004,193,1,90.78,3,1,2,6
C00005,283,1,266.89,2,1,4,7


In [42]:
def join_rfm(x):
    return str(x['R']) + str(x['F']) + str(x['M'])

In [43]:
agregacao['Segmento_RFM'] = agregacao.apply(join_rfm, axis = 1)

In [44]:
agregacao.head()

,Recenty,Frequency,Monetary,R,F,M,Score_RFM,Segmento_RFM
customer_unique_id,,,,,,,,
C00001,73,1,121.82,4,1,3,8,4.01.03.0
C00002,66,1,155.76,4,1,3,8,4.01.03.0
C00003,337,1,181.55,2,1,3,6,2.01.03.0
C00004,193,1,90.78,3,1,2,6,3.01.02.0
C00005,283,1,266.89,2,1,4,7,2.01.04.0


In [45]:
agregacao['Segmento_RFM'] = agregacao.apply(join_rfm, axis = 1)

In [46]:
agregacao.head()

,Recenty,Frequency,Monetary,R,F,M,Score_RFM,Segmento_RFM
customer_unique_id,,,,,,,,
C00001,73,1,121.82,4,1,3,8,413
C00002,66,1,155.76,4,1,3,8,413
C00003,337,1,181.55,2,1,3,6,213
C00004,193,1,90.78,3,1,2,6,312
C00005,283,1,266.89,2,1,4,7,214


In [47]:
rfm_count_unique = agregacao.groupby('Segmento_RFM')['Segmento_RFM'].nunique()

In [48]:
print(rfm_count_unique.count())

32


In [49]:
rfm = agregacao

In [50]:
rfm.head()

,Recenty,Frequency,Monetary,R,F,M,Score_RFM,Segmento_RFM
customer_unique_id,,,,,,,,
C00001,73,1,121.82,4,1,3,8,413
C00002,66,1,155.76,4,1,3,8,413
C00003,337,1,181.55,2,1,3,6,213
C00004,193,1,90.78,3,1,2,6,312
C00005,283,1,266.89,2,1,4,7,214


In [51]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94087 entries, C00001 to C96999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Recenty       94087 non-null  int64  
 1   Frequency     94087 non-null  int64  
 2   Monetary      94087 non-null  float64
 3   R             94087 non-null  int64  
 4   F             94087 non-null  int64  
 5   M             94087 non-null  int64  
 6   Score_RFM     94087 non-null  int64  
 7   Segmento_RFM  94087 non-null  object 
dtypes: float64(1), int64(6), object(1)
memory usage: 8.5+ MB


In [52]:
rfm['Segmento_RFM'] = rfm['Segmento_RFM'].astype(str).astype(int)

In [53]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94087 entries, C00001 to C96999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Recenty       94087 non-null  int64  
 1   Frequency     94087 non-null  int64  
 2   Monetary      94087 non-null  float64
 3   R             94087 non-null  int64  
 4   F             94087 non-null  int64  
 5   M             94087 non-null  int64  
 6   Score_RFM     94087 non-null  int64  
 7   Segmento_RFM  94087 non-null  int32  
dtypes: float64(1), int32(1), int64(6)
memory usage: 8.1+ MB


In [54]:
rfm.describe()

,Recenty,Frequency,Monetary,R,F,M,Score_RFM,Segmento_RFM
count,94087.000000,94087.000000,94087.000000,94087.000000,94087.000000,94087.000000,94087.000000,94087.000000
mean,243.803575,1.239077,214.249054,2.503895,1.150627,2.499283,6.153804,264.395081
std,153.156983,0.850594,647.368039,1.119577,0.357686,1.118482,1.715630,112.037091
min,1.000000,1.000000,9.590000,1.000000,1.000000,1.000000,3.000000,111.000000
25%,120.000000,1.000000,64.000000,2.000000,1.000000,1.000000,5.000000,211.000000
50%,224.000000,1.000000,113.150000,3.000000,1.000000,2.000000,6.000000,311.000000
75%,353.000000,1.000000,203.770000,4.000000,1.000000,3.000000,7.000000,411.000000
max,730.000000,75.000000,109312.640000,4.000000,2.000000,4.000000,10.000000,424.000000


In [55]:
def rfm_level(df):
    
    if (df['Segmento_RFM'] >= 424 | (df['Score_RFM'] >= 9)) :
        return 'Clientes VIP'
    
    elif ((df['Score_RFM'] >= 8) & (df['M'] == 4)):
        return 'Clientes Leais Que Compram com Frequência'
    
    elif ((df['Score_RFM'] >= 6) & (df['F'] >= 2)):
        return 'Clientes Leais'
    
    elif ((df['Score_RFM'] <= 4) & (df['R'] == 1)):
        return 'Clientes Quase Perdidos'
    
    elif ((df['Segmento_RFM'] >= 221) | (df['Score_RFM'] >= 6)):
        return 'Potenciais Clientes Leais'
    
    elif ((df['Segmento_RFM'] >= 121) & (df['R'] == 1) | (df['Score_RFM'] == 5)):
        return 'Clientes Que Precisam de Atenção'
    
    else:
        return 'Clientes Perdidos'

In [56]:
def rfm_action(df):
    
    if (df['Segmento_RFM'] >= 424 | (df['Score_RFM'] >= 9)) :
        return 'Incentivos não relacionados a preços; Oferecer edição limitada e programas de fidelidade'
    
    elif ((df['Score_RFM'] >= 8) & (df['M'] == 4)):
        return 'Oferecer itens mais caros (Upsell)'
    
    elif ((df['Score_RFM'] >= 6) & (df['F'] >= 2)):
        return 'Oferecer programas de fidelidade e venda cruzada (Cross-Sell)'
    
    elif ((df['Score_RFM'] <= 4) & (df['R'] == 1)):
        return 'Oferecer Incentivos de preços agressivos'
    
    elif ((df['Segmento_RFM'] >= 221) | (df['Score_RFM'] >= 6)):
        return 'Recomendações de venda cruzada e cupons de desconto'
    
    elif (((df['Segmento_RFM'] >= 121) & (df['R'] == 1)) | (df['Score_RFM'] == 5)):
        return 'Incentivos de preço e oferta por tempo limitado'
    
    else:
        return 'Não gaste muito tentando readquirir esse cliente'

In [57]:
rfm['Segmento de Cliente'] = rfm.apply(rfm_level, axis = 1)

In [58]:
rfm['Marketing Action'] = rfm.apply(rfm_action, axis = 1)

In [59]:
rfm.head(10)

,Recenty,Frequency,Monetary,R,F,M,Score_RFM,Segmento_RFM,Segmento de Cliente,Marketing Action
customer_unique_id,,,,,,,,,,
C00001,73,1,121.82,4,1,3,8,413,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00002,66,1,155.76,4,1,3,8,413,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00003,337,1,181.55,2,1,3,6,213,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00004,193,1,90.78,3,1,2,6,312,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00005,283,1,266.89,2,1,4,7,214,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00006,105,2,267.96,4,2,4,10,424,Clientes Leais Que Compram com Frequência,Oferecer itens mais caros (Upsell)
C00007,82,1,70.03,4,1,2,7,412,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00008,169,1,165.94,3,1,3,7,313,Potenciais Clientes Leais,Recomendações de venda cruzada e cupons de des...
C00009,366,1,103.64,1,1,2,4,112,Clientes Quase Perdidos,Oferecer Incentivos de preços agressivos


In [60]:
rfm['Segmento de Cliente'].value_counts()

Segmento de Cliente
Potenciais Clientes Leais                    45421
Clientes Leais Que Compram com Frequência    14249
Clientes Quase Perdidos                      11595
Clientes Que Precisam de Atenção             11043
Clientes Leais                                6141
Clientes Perdidos                             5638
Name: count, dtype: int64

In [61]:
rfm_level_agg = rfm.groupby(['Segmento de Cliente']).agg({'Recenty': 'mean',
                                                          'Frequency': 'mean',
                                                          'Monetary': ['mean', 'count'],
                                                          'Marketing Action': 'unique'}).round(1)

In [62]:
rfm_level_agg

Recenty Frequency Monetary         \
                                             mean      mean     mean  count   
Segmento de Cliente                                                           
Clientes Leais                              305.2       2.4    356.9   6141   
Clientes Leais Que Compram com Frequência   146.1       1.9    619.3  14249   
Clientes Perdidos                           281.9       1.0     44.2   5638   
Clientes Quase Perdidos                     457.3       1.0     64.2  11595   
Clientes Que Precisam de Atenção            368.5       1.0    114.9  11043   
Potenciais Clientes Leais                   176.6       1.0    151.5  45421   

                                                                            Marketing Action  
                                                                                      unique  
Segmento de Cliente                                                                           
Clientes Leais                             [Oferecer programas de fidelidade e venda cruz...  
Clientes Leais Que Compram com Frequência               [Oferecer itens mais caros (Upsell)]  
Clientes Perdidos                          [Não gaste muito tentando readquirir esse clie...  
Clientes Quase Perdidos                           [Oferecer Incentivos de preços agressivos]  
Clientes Que Precisam de Atenção           [Incentivos de preço e oferta por tempo limitado]  
Potenciais Clientes Leais                  [Recomendações de venda cruzada e cupons de de...

In [63]:
rfm_level_agg.to_csv("rfm.csv")

In [64]:
fig = go.Figure(go.Treemap(
    labels = rfm_level_agg.index, 
    parents = [''] * len(rfm_level_agg.index),
    values = rfm_level_agg[('Monetary', 'count')]
))

fig.update_layout(
    title="Distribuição de Clientes por Segmento"
)

fig.show()